# Chapter 7 인공지능 프로그래밍 Ⅰ 실습, 문제 정의에서 결과 해석까지

**AI교육의 이해 / 언플러그드AI교육, Chapter 7 인공지능 프로그래밍**
전인성, 광주교육대학교 컴퓨터교육과

<a href="https://colab.research.google.com/github/jinsung4069/jinsung4069.github.io/blob/main/data/lectures/ai-education-ch7-1-model-lab.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

## 상황
Chapter 6에서 다듬은 펭귄 관측 자료로 세 가지 모델을 만듭니다. 엔트리의 `분류: 숫자`, `예측: 숫자`, `군집: 숫자` 모델과 같은 일을 파이썬 코드로 수행하며, 각 단계에서 사람이 내리는 결정을 확인합니다.

| 모델 | 엔트리 모델 | 이 실습의 질문 |
|---|---|---|
| 분류 | 분류: 숫자 (k-최근접 이웃) | 부리 길이와 날개 길이로 종을 맞힐 수 있을까? |
| 예측 | 예측: 숫자 (선형 회귀) | 날개 길이로 몸무게를 얼마나 예측할 수 있을까? |
| 군집 | 군집: 숫자 (k-평균) | 종 이름 없이 묶으면 종과 비슷한 무리가 생길까? |

## 실습 방법
1. 위에서부터 코드 셀을 하나씩 실행합니다. 코드 안의 `____` 는 직접 채우는 빈칸이며 다섯 곳입니다.
2. **예상 기록** 칸은 결과를 보기 전에 먼저 적습니다.
3. 기록을 저장하려면 `파일 > Drive에 사본 저장` 을 먼저 선택합니다.

## 사용하는 데이터
Palmer Station LTER 펭귄 관측 자료(Gorman 외, 2014)의 정리본(palmerpenguins, CC0)입니다. Chapter 6 실습에서 넣었던 표기 차이, 단위 오류, 중복 행이 없는 원래 자료를 불러온 뒤 측정값이 빈 2행만 지웁니다.

## 0단계, 준비 코드 실행

In [ ]:
# @title 준비 코드 (실행만 하세요) { display-mode: "form" }
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython import get_ipython
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LinearRegression
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, confusion_matrix, r2_score, mean_absolute_error

def _blank_handler(self, etype, value, tb, tb_offset=None):
    history = self.user_ns.get('In') or ['']
    if '____' in history[-1]:
        print('[빈칸 확인] 아직 채우지 않은 빈칸 ____ 이 있습니다. 빈칸을 채운 뒤 다시 실행하세요.')
        return None
    return self.showtraceback((etype, value, tb), tb_offset=tb_offset)

get_ipython().set_custom_exc((Exception,), _blank_handler)

SEED = 2026
pd.set_option('display.precision', 3)
DATA_URL = 'https://raw.githubusercontent.com/jinsung4069/jinsung4069.github.io/main/data/lectures/ai-education-penguins.csv'
MEASURES = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']
SPECIES = ['Adelie', 'Chinstrap', 'Gentoo']
COLORS = {'Adelie': '#e07b39', 'Chinstrap': '#8e5bb5', 'Gentoo': '#2a8c82'}

def check(n, value):
    ok = {
        1: lambda v: 0.1 <= float(v) <= 0.4,
        2: lambda v: 1 <= int(v) <= 30,
        3: lambda v: v == 'body_mass_g',
        4: lambda v: v == 'body_mass_g',
        5: lambda v: int(v) == 3,
    }
    try:
        passed = ok[n](value)
    except Exception:
        passed = False
    print(f'[빈칸 {n}] ' + ('확인했습니다. 다음 단계로 넘어가세요.' if passed else '결과가 예상과 다릅니다. 안내 문장을 다시 읽고 빈칸을 고쳐 보세요.'))

def plot_confusion(cm, labels, title):
    plt.figure(figsize=(4.2, 3.6))
    plt.imshow(cm, cmap='Blues')
    for i in range(len(labels)):
        for j in range(len(labels)):
            plt.text(j, i, cm[i, j], ha='center', va='center', color='white' if cm[i, j] > cm.max() / 2 else 'black')
    plt.xticks(range(len(labels)), labels); plt.yticks(range(len(labels)), labels)
    plt.xlabel('predicted'); plt.ylabel('actual'); plt.title(title); plt.show()

## 1단계, 문제 정의
모델을 만들기 전에 입력, 출력, 학습 방식, 성공 기준을 정합니다. 이 표가 없으면 결과가 좋은지 나쁜지 판단할 수 없습니다.

### 예상 기록 (코드를 실행하기 전에 적기)
| 항목 | 분류 모델 기록 |
|---|---|
| 입력 (특징) | 부리 길이, 날개 길이 |
| 출력 (정답) | 종 |
| 학습 방식 | 지도학습 |
| 성공 기준 (평가 자료 정확도 몇 % 이상?) | |
| 틀렸을 때의 영향과 대처 | |

## 2단계, 데이터 준비
특징(X)과 정답(y)을 고르고, 학습 자료와 평가 자료를 나눕니다. 평가 자료는 학습에 쓰지 않고 마지막 확인에만 씁니다.

In [ ]:
df = pd.read_csv(DATA_URL).dropna(subset=MEASURES).reset_index(drop=True)
print(f'자료 {len(df)}행')
FEATURES = ['bill_length_mm', 'flipper_length_mm']
X = df[FEATURES]
y = df['species']
display(df[FEATURES + ['species']].head())

In [ ]:
# 빈칸 1, 평가 자료의 비율을 0.1과 0.4 사이로 적습니다. 엔트리의 기본값은 0.25입니다.
test_size = ____

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, stratify=y, random_state=SEED)
print(f'학습 자료 {len(X_train)}행, 평가 자료 {len(X_test)}행')
display(pd.DataFrame({'학습': y_train.value_counts(), '평가': y_test.value_counts()}))
check(1, test_size)

## 3단계, 모델 실행 (분류, k-최근접 이웃)
새 펭귄과 가장 가까운 학습 자료 k마리의 종을 보고 다수결로 판단합니다.

### 예상 기록
- k를 1로 하면 학습 자료 정확도와 평가 자료 정확도는 각각 어떻게 될까요?

In [ ]:
# 빈칸 2, 참고할 이웃의 수 k를 1부터 30 사이의 정수로 적습니다. 교재의 엔트리 예시는 10입니다.
k = ____

model = KNeighborsClassifier(n_neighbors=k).fit(X_train, y_train)
pred = model.predict(X_test)
print(f'k = {k}')
print(f'학습 자료 정확도 {accuracy_score(y_train, model.predict(X_train)):.1%}')
print(f'평가 자료 정확도 {accuracy_score(y_test, pred):.1%}  ({(pred == y_test).sum()} / {len(y_test)})')
plot_confusion(confusion_matrix(y_test, pred, labels=SPECIES), SPECIES, f'kNN (k={k}), test data')
check(2, k)

## 4단계, 결과 해석
### 4-1. 어떤 펭귄을 틀렸을까

In [ ]:
wrong = X_test[pred != y_test].assign(actual=y_test[pred != y_test], predicted=pred[pred != y_test])
display(wrong)

### 4-2. 새 펭귄의 예측과 신뢰도
엔트리의 `신뢰도` 블록처럼, 이웃 k마리 중 각 종의 비율을 확인합니다.

In [ ]:
new = pd.DataFrame({'bill_length_mm': [39.0, 46.0, 50.0], 'flipper_length_mm': [190, 205, 196]})
proba = pd.DataFrame(model.predict_proba(new), columns=model.classes_).round(2)
display(pd.concat([new, proba], axis=1).assign(predicted=model.predict(new)))

### 4-3. k를 바꾸면
k를 1부터 25까지 바꾸며 학습 자료와 평가 자료의 정확도를 비교합니다.

In [ ]:
rows = []
for kk in range(1, 26):
    m = KNeighborsClassifier(n_neighbors=kk).fit(X_train, y_train)
    rows.append({'k': kk, 'train': accuracy_score(y_train, m.predict(X_train)), 'test': accuracy_score(y_test, m.predict(X_test))})
curve = pd.DataFrame(rows).set_index('k')
curve.plot(marker='o', ylim=(0.8, 1.01), figsize=(6.5, 3.5))
plt.ylabel('accuracy'); plt.title('Accuracy by k'); plt.show()
display(curve.loc[[1, 3, 5, 10, 15, 25]].style.format('{:.1%}'))

### 4-4. 단위가 다른 특징을 더하면
몸무게(g)는 부리 길이(mm)보다 숫자가 백 배쯤 큽니다. 거리로 이웃을 찾는 모델에 몸무게를 더하면 어떻게 될까요? 크기를 맞추는 표준화도 함께 비교합니다.

In [ ]:
# 빈칸 3, 특징에 더할 몸무게 열의 이름을 적습니다.
extra = '____'

cols = FEATURES + [extra]
Xa_train, Xa_test = df.loc[X_train.index, cols], df.loc[X_test.index, cols]
raw = KNeighborsClassifier(n_neighbors=k).fit(Xa_train, y_train)
scaled = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k)).fit(Xa_train, y_train)
print(f'두 특징 (표준화 없음)        {accuracy_score(y_test, model.predict(X_test)):.1%}')
print(f'세 특징, 표준화 없음        {accuracy_score(y_test, raw.predict(Xa_test)):.1%}')
print(f'세 특징, 표준화 후          {accuracy_score(y_test, scaled.predict(Xa_test)):.1%}')
check(3, extra)

### 해석 기록
- k가 1일 때 학습 자료 정확도가 100%에 가까운 이유는 무엇인가요? 이 값을 모델의 성능으로 보고하면 안 되는 이유는?
- 몸무게를 표준화 없이 더했을 때 정확도가 떨어진 이유를 거리 계산과 연결해 설명해 보세요.

## 5단계, 예측 모델 (선형 회귀)
날개 길이로 몸무게를 예측합니다. 결과는 직선의 식, 결정계수(R²), 평균 절대 오차로 해석합니다.

In [ ]:
# 빈칸 4, 예측할 값이 담긴 열의 이름을 적습니다. 몸무게 열입니다.
target = '____'

Xr = df[['flipper_length_mm']]
yr = df[target]
Xr_train, Xr_test, yr_train, yr_test = train_test_split(Xr, yr, test_size=0.25, random_state=SEED)
reg = LinearRegression().fit(Xr_train, yr_train)
yr_pred = reg.predict(Xr_test)
print(f'몸무게 = {reg.coef_[0]:.1f} x 날개 길이 + ({reg.intercept_:.0f})')
print(f'평가 자료 결정계수 R² {r2_score(yr_test, yr_pred):.2f}')
print(f'평가 자료 평균 절대 오차 {mean_absolute_error(yr_test, yr_pred):.0f} g')
check(4, target)

plt.figure(figsize=(6, 4))
plt.scatter(Xr_train, yr_train, s=12, alpha=0.5, label='train')
plt.scatter(Xr_test, yr_test, s=14, alpha=0.8, label='test')
line = np.linspace(170, 235, 50).reshape(-1, 1)
plt.plot(line, reg.predict(pd.DataFrame(line, columns=['flipper_length_mm'])), color='black')
plt.xlabel('flipper_length_mm'); plt.ylabel('body_mass_g'); plt.legend(); plt.title('Linear regression'); plt.show()

### 해석 기록
- 기울기의 뜻을 단위를 넣어 한 문장으로 적어 보세요.
- 날개 길이가 260mm인 펭귄의 몸무게도 이 식으로 예측해도 될까요?

## 6단계, 군집 모델 (k-평균)
종 이름을 주지 않고 부리 길이와 부리 두께만으로 펭귄을 묶습니다. 묶은 뒤에 실제 종과 비교합니다.

In [ ]:
# 빈칸 5, 만들 군집의 수를 적습니다. 세 종과 비교하려고 합니다.
n_clusters = ____

Xc = StandardScaler().fit_transform(df[['bill_length_mm', 'bill_depth_mm']])
km = KMeans(n_clusters=n_clusters, n_init=10, random_state=SEED).fit(Xc)
df['cluster'] = km.labels_
display(pd.crosstab(df['cluster'], df['species']))
check(5, n_clusters)

plt.figure(figsize=(6, 4.5))
for c in sorted(df['cluster'].unique()):
    part = df[df['cluster'] == c]
    plt.scatter(part['bill_length_mm'], part['bill_depth_mm'], s=16, alpha=0.75, label=f'cluster {c}')
plt.xlabel('bill_length_mm'); plt.ylabel('bill_depth_mm'); plt.legend(); plt.title('k-means clusters'); plt.show()

### 해석 기록
- 군집 번호는 종 이름과 어떤 관계가 있나요? 번호 자체에 의미가 있나요?
- 실제 종과 다르게 묶인 펭귄은 어느 종과 어느 종 사이에 있나요?

## 7단계, 실행 기록
| 항목 | 기록 |
|---|---|
| 빈칸에 넣은 값 (평가 비율, k, 군집 수) | |
| 분류 모델의 평가 자료 정확도와 자주 틀린 종 | |
| 예측 모델의 식, R², 평균 절대 오차 | |
| 군집 결과와 실제 종의 일치 정도 | |
| 1단계 성공 기준의 달성 여부 | |
| 이 모델을 쓸 수 없는 상황 | |

---
자료 출처: Gorman, K. B., Williams, T. D., & Fraser, W. R. (2014). PLoS ONE, 9(3), e90081. Horst, A. M., Hill, A. P., & Gorman, K. B. (2020). palmerpenguins, CC0.